In [33]:
#libraries
import QuantLib as ql 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from scipy.integrate import quad

## QuantLib implementation

In [34]:
def heston_quantlib_prices(
    S0, #stock price today
    K, #strike price
    T, #time to maturity (in years)
    r, #risk free interest rate
    v0, #variance today
    kappa, #speed of mean reversion
    theta, #long run variance 
    sigma, #volatility of variance 
    rho, #correlation between stock and variance 
    q=0.0 #dividend yield
):
    #date setup
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today 

    #conversion of T into maturity date
    maturity_date = today + int(round(T * 365))

    #day count conversion
    day_count = ql.Actual365Fixed()

    #spot price
    spot = ql.QuoteHandle(
        ql.SimpleQuote(S0)
    )

    #risk free rate 
    risk_free_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(today,r,day_count)
    )

    dividend_curve = ql.YieldTermStructureHandle(
        ql.FlatForward(today,q,day_count)
    )

    heston_process = ql.HestonProcess(
        risk_free_curve,
        dividend_curve,
        spot,
        v0,
        kappa,
        theta,
        sigma,
        rho 
    )

    #heston model 
    heston_model = ql.HestonModel(heston_process)

    #analytical heston pricing engine
    engine = ql.AnalyticHestonEngine(heston_model)

    #European call
    payoff = ql.PlainVanillaPayoff(
        ql.Option.Call,
        K 
    )

    #
    exercise = ql.EuropeanExercise(
        maturity_date
    )

    option = ql.VanillaOption(
        payoff,
        exercise
    )

    option.setPricingEngine(engine)

    return option.NPV()


## Scipy implementation

In [35]:
def heston_charac_function(
    u, 
    S0, 
    T, 
    r, 
    v0, 
    kappa,
    theta,
    sigma, 
    rho
): 
    i = 1j 
    d = np.sqrt(
        (rho*sigma*i*u-kappa)**2 
        + sigma**2 * (i * u + u**2)
    )
    g = (
        kappa - rho * sigma * i * u - d
    )/(
        kappa - rho * sigma * i * u + d
    )
    C = (
        r * i * u * T 
        + (kappa * theta / sigma**2)
        *(
            (kappa - rho * sigma * i * u - d) * T 
            - 2 
            * np.log(
                (1-g * np.exp(-d*T))
                / (1-g)
            )
        )
    )
    D = (
        (kappa - rho * sigma * i * u - d) 
        / sigma ** 2 
        * (
            (1 - np.exp(-d * T))
            / (1 - g * np.exp(-d * T))
        )
    )
    return np.exp(
        C 
        + D * v0 
        + i * u * np.log(S0)
    ) 

def heston_probability(
        j, 
        S0, 
        K, 
        T, 
        r, 
        v0, 
        kappa, 
        theta, 
        sigma, 
        rho
): 
    phi_minus_i = heston_charac_function(
        -1j, 
        S0, 
        T, 
        r, 
        v0, 
        kappa, 
        theta, 
        sigma, 
        rho
    )
    def integrand(u):
        phi_u = heston_charac_function(
            u, 
            S0, 
            T, 
            r, 
            v0, 
            kappa, 
            theta, 
            sigma, 
            rho
        )
        if j == 1:
            phi_shifted = heston_charac_function(
                u-1j, 
                S0, 
                T, 
                r, 
                v0, 
                kappa, 
                theta, 
                sigma, 
                rho
            )
            value = (
                np.exp(-1j * u * np.log(K))
                * phi_shifted
                / (1j * u * phi_minus_i)
            )
        else: 
            value = (
                np.exp(-1j * u * np.log(K))
                * phi_u
                /(1j * u)
            )
        return np.read(value)
    integral, error = quad(
        integrand, 
        1e-8, 
        100, 
        limit=500
    )
    return 0.5 + integral/np.pi 

def heston_price_scipy(
        S0, 
        K, 
        T, 
        r, 
        v0,
        kappa, 
        theta, 
        sigma, 
        rho
): 
    P1 = heston_probability(
        1, 
        S0, 
        K, 
        T, 
        r, 
        v0, 
        kappa, 
        theta, 
        sigma, 
        rho
    )

    P2 = heston_probability(
        2, 
        S0, 
        K, 
        T, 
        r, 
        v0, 
        kappa, 
        theta, 
        sigma, 
        rho
    )

    price = (
        S0 * P1 - K * np.exp(-r * T) * P2
    )

    return price

In [36]:
#test 
price = heston_quantlib_prices(
    S0=100,
    K = 100, 
    T = 1.0, 
    r=0.03, 
    v0 = 0.04, 
    kappa = 2.0, 
    theta=0.04,
    sigma=0.3, 
    rho=-0.70
)

print(f"Option price: {price:.6f}")

Option price: 9.242521
